In [21]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [22]:
# Tool Created 

from langchain_community.tools import tool
import requests


@tool
def get_conversion_factor(base_currency:str, target_currency:str) -> float:
    """
    this function fetches the currency conversion factor between a given base currency and a target currency.
    """

    url = f"https://v6.exchangerate-api.com/v6/eaa7653674a33010eda5fc73/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()


@tool
def convert_currency(base_currency_value:int, conversion_rate:Annotated[float, InjectedToolArg]) -> float:
    """
    this function converts a given amount from a base currency to a target currency using the conversion factor.
    """

    return base_currency_value * conversion_rate

In [23]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1788393601,
 'time_last_update_utc': 'Thu, 03 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1788480001,
 'time_next_update_utc': 'Fri, 04 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.8986}

In [24]:
convert_currency.invoke({'base_currency_value': 10, 'conversion_rate': 82.5})

825.0

In [25]:
## Tool Binding

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [26]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert_currency])

In [27]:
## Tool calling

messages = [HumanMessage("What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr")]

In [28]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [29]:
ai_message = llm_with_tools.invoke(messages)

In [30]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': 'call_1468660',
  'type': 'tool_call'}]

In [ ]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the valur of conversion rate

    if tool_call['name'] == 'get_conversion_factor':
        tool_msg1 = get_conversion_factor.invoke(tool_call)
        print(tool_msg1)

        # fetch conversion rate
        print(tool_msg1.content)['conversion_rate']


    # execute 2nd tool using the conversion rate from tool 1